# 📐 Aula 18 — Intervalo de confiança para a média numa usina solar do Norte de Minas

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte da **equipe de operação de uma usina solar em Janaúba**, no Norte de Minas. A equipe quer saber a **energia média gerada por painel em um dia** ($\mu$, em kWh), mas só consegue medir alguns painéis. O notebook segue a ordem da Aula 18:

1. o **intervalo de confiança** com $\sigma$ conhecido;
2. o que significa **"95% de confiança"**;
3. o que muda a **amplitude** do intervalo;
4. o intervalo com $\sigma$ **desconhecido** (a $t$ de Student);
5. o caso da **amostra grande**.

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude). Confira antes de usar.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores, e os sorteios só saem iguais aos do texto se a ordem for respeitada.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.235` = 0,235).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**;
- **`scipy.stats`**: as **tabelas** da normal (`norm`) e da t (`t`) dentro do computador. A função `.ppf(p)` dá o número com área $p$ à **esquerda** dele. Exemplos: $z_{0{,}025}$ = `norm.ppf(0.975)` = 1,96 e $t_{0{,}025;9}$ = `t.ppf(0.975, 9)` = 2,262.

Criamos também o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Só o notebook sabe** o valor verdadeiro: a energia por painel tem distribuição normal com $\mu = 5{,}2$ kWh e $\sigma = 0{,}6$ kWh. A equipe não conhece $\mu$ — é isso que ela quer estimar.

In [ ]:
import numpy as np                     # calculadora de números e sorteios
import matplotlib.pyplot as plt        # gráficos
from scipy.stats import norm, t        # as "tabelas"

rng = np.random.default_rng(42)        # sorteador, semente 42
mu, sigma = 5.2, 0.6                   # valores verdadeiros (a equipe NÃO conhece mu)
print("Tudo pronto!")

## 1. O intervalo de confiança com $\sigma$ conhecido

### A situação
O fabricante dos painéis informa que o desvio padrão da energia diária é $\sigma = 0{,}6$ kWh. A equipe mede **$n = 25$** painéis num dia.

### A pergunta
Qual é o **IC de 95%** para a energia média $\mu$?

### A ferramenta da aula
$$IC(\mu;\,1-\alpha) = \left[\bar x - z_{\alpha/2}\,\frac{\sigma}{\sqrt n}\ ;\ \bar x + z_{\alpha/2}\,\frac{\sigma}{\sqrt n}\right]$$
Para 95%: $z_{0{,}025} = 1{,}96$ (na tabela $P(0 \le Z \le z)$, procure 0,475).

### A resposta pela conta
- erro padrão: $\sigma/\sqrt n = 0{,}6/\sqrt{25} = 0{,}12$ kWh;
- margem de erro: $1{,}96 \times 0{,}12 = 0{,}2352$ kWh;
- o intervalo é $\bar x \pm 0{,}2352$. Falta só o $\bar x$, que vem da amostra.

### O que o código faz
1. sorteia a amostra de 25 painéis e calcula $\bar x$;
2. calcula a margem de erro e o intervalo.

In [ ]:
n = 25
amostra = rng.normal(mu, sigma, n)               # energia de 25 painéis (kWh)
xbarra = amostra.mean()
margem = norm.ppf(0.975) * sigma / np.sqrt(n)    # 1,96 × 0,6/√25

print(f"Média da amostra: {xbarra:.3f} kWh")
print(f"Margem de erro:   {margem:.4f} kWh")
print(f"IC de 95%: [{xbarra - margem:.3f} ; {xbarra + margem:.3f}] kWh")

**Como ler:**
- A amostra deu $\bar x \approx 5{,}179$ kWh, e o IC de 95% é cerca de **[4,944 ; 5,414] kWh**.
- Desta vez, o intervalo **contém** o valor verdadeiro $\mu = 5{,}2$. Mas a equipe não sabe disso: na vida real, ela nunca vê $\mu$.
- A pergunta natural é: com que frequência um intervalo construído assim acerta? É a próxima seção.

## 2. O que significa "95% de confiança"?

### A situação
Imagine que a equipe repetisse a medição em **100 dias diferentes**, com 25 painéis por dia, e construísse um IC de 95% em cada dia.

### A pergunta
Quantos desses 100 intervalos contêm o valor verdadeiro $\mu = 5{,}2$?

### A ferramenta da aula
$\mu$ é um **número fixo**; o que muda de amostra para amostra é o **intervalo**. "95% de confiança" quer dizer que cerca de **95% dos intervalos** construídos assim contêm $\mu$ (é a figura dos 20 intervalos da aula).

### O que o código faz
1. sorteia 100 amostras de 25 painéis e constrói os 100 intervalos;
2. conta quantos contêm $\mu$ e desenha os intervalos (em vermelho, os que **não** contêm);
3. repete com **100.000** amostras, para ver a porcentagem com mais precisão.

In [ ]:
medias = rng.normal(mu, sigma, size=(100, n)).mean(axis=1)    # 100 dias, 25 painéis por dia
acertou = np.abs(medias - mu) <= margem
print(f"Intervalos que contêm mu: {acertou.sum()} de 100")

for i, x in enumerate(medias, start=1):
    cor = "#1f3a93" if acertou[i - 1] else "#c0392b"
    plt.plot([x - margem, x + margem], [i, i], color=cor, linewidth=1.5)
plt.axvline(mu, color="black", linestyle="--", label="μ = 5,2 (verdadeiro)")
plt.xlabel("energia por painel (kWh)")
plt.ylabel("dia")
plt.title("100 intervalos de 95%: em vermelho, os que não contêm μ")
plt.legend()
plt.show()

muitas = rng.normal(mu, sigma, size=(100_000, n)).mean(axis=1)
print(f"Com 100.000 amostras: {(np.abs(muitas - mu) <= margem).mean():.1%} dos intervalos contêm mu")

**Como ler:**
- Dos 100 intervalos, **94** contêm $\mu$ e **6** (em vermelho) não contêm. Perto de 95, mas não exatamente: é um sorteio.
- Com 100.000 amostras, a porcentagem fica em **95,0%**: é isso que "95% de confiança" garante.
- Repare que cada intervalo **ou contém $\mu$ ou não contém**. Por isso é errado dizer "há 95% de probabilidade de $\mu$ estar no meu intervalo": os 95% são do **procedimento**, não de um intervalo específico.

## 3. O que muda a amplitude do intervalo?

### A pergunta
Como a **amplitude** do intervalo muda com o **nível de confiança** e com o **tamanho da amostra**?

### A ferramenta da aula
$$\text{Amplitude} = 2\,z_{\alpha/2}\,\frac{\sigma}{\sqrt n}$$
Ela **não depende** de $\bar x$: dá para calcular antes de medir os painéis.

### O que o código faz
Monta uma tabela de amplitudes para $n$ = 16, 25, 64 e 100 e níveis de confiança de 90%, 95% e 99%.

In [ ]:
niveis = [0.90, 0.95, 0.99]
print("   n      90%      95%      99%   (amplitude em kWh)")
for n_ in (16, 25, 64, 100):
    amplitudes = [2 * norm.ppf(1 - (1 - c) / 2) * sigma / np.sqrt(n_) for c in niveis]
    print(f"{n_:4d}   " + "    ".join(f"{a:.3f}" for a in amplitudes))

**Como ler:**
- **Em cada linha**, da esquerda para a direita: mais confiança, intervalo **mais largo** (com $n = 25$: 0,395 → 0,470 → 0,618 kWh).
- **Em cada coluna**, de cima para baixo: mais painéis, intervalo **mais estreito**.
- Compare $n = 16$ com $n = 64$ (4 vezes mais painéis): a amplitude cai **à metade** (0,588 → 0,294 com 95%), porque $\sqrt{64}/\sqrt{16} = 2$.

## 4. E se a equipe não conhecer $\sigma$?

### A situação
Para um modelo novo de painel, o fabricante **não informa** $\sigma$. A equipe mede só **$n = 10$** painéis e calcula o desvio padrão amostral $s$.

### A pergunta
Se a equipe usar $s$ no lugar de $\sigma$ e continuar usando **1,96**, o intervalo ainda acerta 95% das vezes? E com o valor da **tabela t**?

### A ferramenta da aula
Com $\sigma$ desconhecido e população normal:
$$IC(\mu;\,1-\alpha) = \left[\bar x - t_{\alpha/2;\,n-1}\,\frac{s}{\sqrt n}\ ;\ \bar x + t_{\alpha/2;\,n-1}\,\frac{s}{\sqrt n}\right]$$
Com $n = 10$: $\nu = 9$ e $t_{0{,}025;9} = 2{,}262$ (o mesmo do Exemplo 3 da aula, o café de Varginha).

### O que o código faz
1. sorteia 100.000 amostras de 10 painéis e calcula $\bar x$ e $s$ em cada uma;
2. constrói dois intervalos em cada amostra: o **correto** (com 2,262) e o **errado** (com 1,96);
3. conta que porcentagem de cada tipo contém $\mu$.

In [ ]:
n = 10
amostras = rng.normal(mu, sigma, size=(100_000, n))
xb = amostras.mean(axis=1)
s = amostras.std(axis=1, ddof=1)                      # desvio padrão amostral (divide por n - 1)

t_tab = t.ppf(0.975, n - 1)                           # t_{0,025;9} = 2,262
margem_t = t_tab * s / np.sqrt(n)
margem_z = 1.96 * s / np.sqrt(n)

print(f"Valor da tabela t (ν = {n - 1}): {t_tab:.3f}")
print(f"Com a t:    {(np.abs(xb - mu) <= margem_t).mean():.1%} dos intervalos contêm mu")
print(f"Com 1,96:   {(np.abs(xb - mu) <= margem_z).mean():.1%} dos intervalos contêm mu")

**Como ler:**
- Com o valor da **tabela t** (2,262), **94,9%** dos intervalos contêm $\mu$: os 95% prometidos.
- Com **1,96**, só **91,7%**: o intervalo fica estreito demais. Parece mais preciso, mas erra mais do que diz.
- O motivo é o da Aula 17: $s$ muda de amostra para amostra, e a padronização com $s$ segue a $t$, que tem caudas mais pesadas que a normal.

## 5. E com amostra grande?

### A situação
Num mutirão de manutenção, a equipe mede **$n = 50$** painéis do modelo novo (ainda sem conhecer $\sigma$).

### A pergunta
Com amostra grande, ainda faz diferença usar a $t$ ou a normal?

### A ferramenta da aula
Com $n \ge 40$, **não precisamos usar a t-Student**: podemos usar a normal, porque a $t$ e a normal padrão são quase iguais:
$$\bar x \pm z_{\alpha/2}\,\frac{s}{\sqrt n}$$

### O que o código faz
Repete a seção 4 com $n = 50$: 100.000 amostras, intervalos com a $t$ ($t_{0{,}025;49}$) e com 1,96.

In [ ]:
n = 50
amostras = rng.normal(mu, sigma, size=(100_000, n))
xb = amostras.mean(axis=1)
s = amostras.std(axis=1, ddof=1)

t_tab = t.ppf(0.975, n - 1)                           # t_{0,025;49}
print(f"Valor da tabela t (ν = {n - 1}): {t_tab:.3f}   (normal: 1.960)")
print(f"Com a t:    {(np.abs(xb - mu) <= t_tab * s / np.sqrt(n)).mean():.1%} dos intervalos contêm mu")
print(f"Com 1,96:   {(np.abs(xb - mu) <= 1.96 * s / np.sqrt(n)).mean():.1%} dos intervalos contêm mu")

**Como ler:**
- Com $n = 50$, o valor da $t$ é **2,010**, muito perto de 1,96.
- As duas porcentagens ficam perto de 95% (cerca de **94,9%** com a $t$ e **94,3%** com 1,96): a diferença praticamente some.
- Compare com a seção 4: com $n = 10$, usar 1,96 derrubava a confiança para 91,7%. Com amostra grande, usar a normal é aceitável.

## Para fechar

Neste notebook, a equipe da usina solar usou os intervalos de confiança da Aula 18:

- **$\sigma$ conhecido:** $\bar x \pm z_{\alpha/2}\,\sigma/\sqrt n$, com $z$ da tabela normal;
- **$\sigma$ desconhecido e $n$ pequeno:** $\bar x \pm t_{\alpha/2;\,n-1}\,s/\sqrt n$, com a tabela $t$ (população normal);
- **$\sigma$ desconhecido e $n \ge 40$:** $\bar x \pm z_{\alpha/2}\,s/\sqrt n$, porque a $t$ e a normal ficam quase iguais.

"95% de confiança" é a proporção de intervalos que contêm $\mu$ quando o procedimento é repetido muitas vezes. Na **Aula 19**, a mesma ideia vale para a **proporção** e a **variância**, e vamos ver como escolher o **tamanho da amostra**.